In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("="*60)
print("PART 1 & 2: PROCESSING, FEATURE PREPARATION & REGRESSION")
print("="*60)
diabetes = load_diabetes(as_frame=True)
df_reg = diabetes.frame

df_reg['gender_cat'] = np.where(df_reg['sex'] > 0, 'Male', 'Female')
df_reg.iloc[0:5, 0] = np.nan

X_reg = df_reg.drop(columns=['target'])
y_reg = df_reg['target']

numeric_features_reg = X_reg.select_dtypes(include=['int64', 'float64']).columns
categorical_features_reg = ['gender_cat']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))
])

preprocessor_reg = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features_reg),
        ('cat', categorical_transformer, categorical_features_reg)
    ])

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

reg_model = Pipeline(steps=[
    ('preprocessor', preprocessor_reg),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

print("Training Regression Model...")
reg_model.fit(X_train_reg, y_train_reg)

y_pred_reg = reg_model.predict(X_test_reg)

mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

print("\n--- Regression Performance Metrics ---")
print(f"First 5 Predictions: {y_pred_reg[:5].round(2)}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R2 Score): {r2:.3f} (Measures variance explained by model)")


print("\n" + "="*60)
print("="*60)

cancer = load_breast_cancer(as_frame=True)
X_clf = cancer.data
y_clf = cancer.target

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

clf_preprocessor = StandardScaler()

clf_model = Pipeline(steps=[
    ('scaler', clf_preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

print("Training Classification Model...")
clf_model.fit(X_train_clf, y_train_clf)

y_pred_clf = clf_model.predict(X_test_clf)
y_prob_clf = clf_model.predict_proba(X_test_clf)[:, 1]

accuracy = accuracy_score(y_test_clf, y_pred_clf)

print(f"{y_pred_clf[:5]}")
print(f"{y_prob_clf[:5].round(2)}")
print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")

print(classification_report(y_test_clf, y_pred_clf, target_names=cancer.target_names))

print(pd.DataFrame(
    confusion_matrix(y_test_clf, y_pred_clf),
    index=['Actual Malignant', 'Actual Benign'],
    columns=['Predicted Malignant', 'Predicted Benign']
))

PART 1 & 2: PROCESSING, FEATURE PREPARATION & REGRESSION
Training Regression Model...

--- Regression Performance Metrics ---
First 5 Predictions: [147.95 179.66 150.87 253.24 105.15]
Root Mean Squared Error (RMSE): 54.43
R-squared (R2 Score): 0.441 (Measures variance explained by model)

Training Classification Model...
[0 1 0 0 0]
[0.   1.   0.12 0.28 0.01]
Overall Accuracy: 95.61%

              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

                  Predicted Malignant  Predicted Benign
Actual Malignant                   39                 3
Actual Benign                       2                70


In [5]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix

print("--- Step 1: Generating Synthetic Dataset ---")
np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    'Numerical_1': np.random.randn(n_samples) * 10 + 50,
    'Numerical_2': np.random.rand(n_samples) * 100,
    'Categorical_1': np.random.choice(['Low', 'Medium', 'High'], size=n_samples),
    'Categorical_2': np.random.choice(['Type_A', 'Type_B'], size=n_samples),
    'Missing_Feature': np.random.choice([np.nan, 1.0, 2.0, 3.0], size=n_samples, p=[0.1, 0.3, 0.3, 0.3])
})

data['Regression_Target'] = data['Numerical_1'] * 2.5 + data['Numerical_2'] * 0.8 + np.random.randn(n_samples) * 5
data['Classification_Target'] = (data['Regression_Target'] > data['Regression_Target'].median()).astype(int)

print(f"Dataset shape: {data.shape}\n")



num_features = ['Numerical_1', 'Numerical_2', 'Missing_Feature']
cat_features = ['Categorical_1', 'Categorical_2']

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ])


X_reg = data.drop(columns=['Regression_Target', 'Classification_Target'])
y_reg = data['Regression_Target']
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)


reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

reg_pipeline.fit(X_train_reg, y_train_reg)

y_pred_reg = reg_pipeline.predict(X_test_reg)

mse = mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"MSE: {mse:.4f}")
print(f"Regression R² Score: {r2:.4f}")



X_clf = data.drop(columns=['Regression_Target', 'Classification_Target'])
y_clf = data['Classification_Target']
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

clf_pipeline.fit(X_train_clf, y_train_clf)

y_pred_clf = clf_pipeline.predict(X_test_clf)

accuracy = accuracy_score(y_test_clf, y_pred_clf)
print(f"Classification Accuracy: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test_clf, y_pred_clf))

print("Confusion Matrix:")
print(confusion_matrix(y_test_clf, y_pred_clf))

--- Step 1: Generating Synthetic Dataset ---
Dataset shape: (1000, 7)

MSE: 23.9083
Regression R² Score: 0.9800
Classification Accuracy: 94.50%

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.96      0.94        97
           1       0.96      0.93      0.95       103

    accuracy                           0.94       200
   macro avg       0.95      0.95      0.94       200
weighted avg       0.95      0.94      0.95       200

Confusion Matrix:
[[93  4]
 [ 7 96]]


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from google.colab import files
uploaded=files.upload()
# 1. Load Dataset
sana = pd.read_csv("Cleaned_Indian_Employee_Data.csv")

# Clean infinity and missing values from numeric feature columns immediately
numeric_features = ["Age", "Experience_Years", "Projects_Completed", "Overtime_Hours", "Salary", "Performance_Rating"]
sana[numeric_features] = sana[numeric_features].replace([np.inf, -np.inf], np.nan)
sana = sana.dropna(subset=numeric_features)

# 2. Data Processing & Feature Preparation
encoder = LabelEncoder()
categorical_columns = ["Gender", "Department", "Position", "City", "State", "Education"]
for col in categorical_columns:
    sana[col] = encoder.fit_transform(sana[col])

# Regression Target & Features
X_reg = sana[["Age", "Experience_Years", "Projects_Completed", "Overtime_Hours"]]
y_reg = sana["Salary"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

reg_scaler = StandardScaler()
X_train_reg = reg_scaler.fit_transform(X_train_reg)
X_test_reg = reg_scaler.transform(X_test_reg)

# Classification Target & Features
sana["Performance"] = np.where(sana["Performance_Rating"] >= 4, 1, 0)
X = sana.drop(["Employee_ID", "Name", "Joining_Date", "Performance_Rating", "Performance"], axis=1)
y = sana["Performance"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

clf_scaler = StandardScaler()
X_train = clf_scaler.fit_transform(X_train)
X_test = clf_scaler.transform(X_test)

# 3. Model Training & 4. Predictive Outputs and Performance

# --- Regression Model ---
regresson = LinearRegression()
regresson.fit(X_train_reg, y_train_reg)
salary_prediction = regresson.predict(X_test_reg)

print("=== Regression Evaluation ===")
print("MAE :", mean_absolute_error(y_test_reg, salary_prediction))
print("MSE :", mean_squared_error(y_test_reg, salary_prediction))
print("R2 Score :", r2_score(y_test_reg, salary_prediction))

# --- Logistic Regression ---
log_model = LogisticRegression()
log_model.fit(X_train, y_train)
log_prediction = log_model.predict(X_test)

print("\n========== Logistic Regression ==========")
print("Accuracy :", accuracy_score(y_test, log_prediction))
print("Precision :", precision_score(y_test, log_prediction))
print("Recall :", recall_score(y_test, log_prediction))
print("F1 Score :", f1_score(y_test, log_prediction))
print("Confusion Matrix:\n", confusion_matrix(y_test, log_prediction))
print("\nClassification Report:\n", classification_report(y_test, log_prediction))

# --- Decision Tree ---
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
tree_prediction = tree.predict(X_test)

print("\n========== Decision Tree ==========")
print("Accuracy :", accuracy_score(y_test, tree_prediction))

# --- Random Forest ---
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)
forest_prediction = forest.predict(X_test)

print("\n========== Random Forest ==========")
print("Accuracy :", accuracy_score(y_test, forest_prediction))

# Feature Importance
importance = pd.DataFrame({"Feature": X.columns, "Importance": forest.feature_importances_})
importance = importance.sort_values(by="Importance", ascending=False)
print("\nMost Important Features:\n", importance)

# Save Outputs
joblib.dump(forest, "employee_prediction_model.pkl")
joblib.dump(clf_scaler, "standard_scaler.pkl")
print("\nModel Saved Successfully")

sana["Prediction"] = forest.predict(clf_scaler.transform(X))
top = sana[sana["Prediction"] == 1]
print("\nEmployees Predicted as High Performance:\n", top[["Employee_ID", "Name", "Department", "Salary", "Performance_Rating"]])

# Increment Recommendation Workflow
def calculate_increment(rating):
    if rating >= 4.8: return "15 %"
    elif rating >= 4.5: return "10 %"
    elif rating >= 4.0: return "7 %"
    elif rating >= 3.5: return "5 %"
    else: return "No Increment"

sana["Recommended Increment"] = sana["Performance_Rating"].apply(calculate_increment)
print("\nIncrement Recommendation:\n", sana[["Employee_ID", "Name", "Performance_Rating", "Recommended Increment"]])

sana.to_csv("Prediction_Report.csv", index=False)
print("\nPrediction Report Saved Successfully")

# Deployment Inference Example
loaded_model = joblib.load("employee_prediction_model.pkl")
loaded_scaler = joblib.load("standard_scaler.pkl")

sample = [[30, 1, 2, 65000, 5, 1, 1, 3, 4, 10, 40]]
sample_scaled = loaded_scaler.transform(sample)
result = loaded_model.predict(sample_scaled)

if result[0] == 1:
    print("\nNew Employee Evaluation: High Performance Employee")
else:
    print("\nNew Employee Evaluation: Average Performance Employee")

Saving Cleaned_Indian_Employee_Data.csv to Cleaned_Indian_Employee_Data.csv
=== Regression Evaluation ===
MAE : 267140.7208566753
MSE : 123560984215.61038
R2 Score : 0.23400883626069025

========== Logistic Regression ==========
Accuracy : 0.875
Precision : 0.8571428571428571
Recall : 1.0
F1 Score : 0.9230769230769231
Confusion Matrix:
 [[1 1]
 [0 6]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       0.86      1.00      0.92         6

    accuracy                           0.88         8
   macro avg       0.93      0.75      0.79         8
weighted avg       0.89      0.88      0.86         8


========== Decision Tree ==========
Accuracy : 0.75

========== Random Forest ==========
Accuracy : 0.75

Most Important Features:
                Feature  Importance
3             Position    0.196637
10      Overtime_Hours    0.166928
4               Salary    0.128604
9   Projects_Complete

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# 1. Load Dataset
df = pd.read_csv("Cleaned_Indian_Employee_Data.csv")

# Clean infinity and missing values from numeric feature columns immediately
numeric_features = ["Age", "Experience_Years", "Projects_Completed", "Overtime_Hours", "Salary", "Performance_Rating"]
df[numeric_features] = df[numeric_features].replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=numeric_features)

# 2. Data Processing & Feature Preparation
encoder = LabelEncoder()
categorical_columns = ["Gender", "Department", "Position", "City", "State", "Education"]
for col in categorical_columns:
    df[col] = encoder.fit_transform(df[col])

# Regression Target & Features
X_reg = df[["Age", "Experience_Years", "Projects_Completed", "Overtime_Hours"]]
y_reg = df["Salary"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

reg_scaler = StandardScaler()
X_train_reg = reg_scaler.fit_transform(X_train_reg)
X_test_reg = reg_scaler.transform(X_test_reg)

# Classification Target & Features
df["Performance"] = np.where(df["Performance_Rating"] >= 4, 1, 0)
X = df.drop(["Employee_ID", "Name", "Joining_Date", "Performance_Rating", "Performance"], axis=1)
y = df["Performance"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

clf_scaler = StandardScaler()
X_train = clf_scaler.fit_transform(X_train)
X_test = clf_scaler.transform(X_test)

# 3. Model Training & 4. Predictive Outputs and Performance

# --- Regression Model ---
regresson = LinearRegression()
regresson.fit(X_train_reg, y_train_reg)
salary_prediction = regresson.predict(X_test_reg)

print("=== Regression Evaluation ===")
print("MAE :", mean_absolute_error(y_test_reg, salary_prediction))
print("MSE :", mean_squared_error(y_test_reg, salary_prediction))
print("R2 Score :", r2_score(y_test_reg, salary_prediction))

# --- Logistic Regression ---
log_model = LogisticRegression()
log_model.fit(X_train, y_train)
log_prediction = log_model.predict(X_test)

print("\n========== Logistic Regression ==========")
print("Accuracy :", accuracy_score(y_test, log_prediction))
print("Precision :", precision_score(y_test, log_prediction))
print("Recall :", recall_score(y_test, log_prediction))
print("F1 Score :", f1_score(y_test, log_prediction))
print("Confusion Matrix:\n", confusion_matrix(y_test, log_prediction))
print("\nClassification Report:\n", classification_report(y_test, log_prediction))

# --- Decision Tree ---
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
tree_prediction = tree.predict(X_test)

print("\n========== Decision Tree ==========")
print("Accuracy :", accuracy_score(y_test, tree_prediction))

# --- Random Forest ---
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)
forest_prediction = forest.predict(X_test)

print("\n========== Random Forest ==========")
print("Accuracy :", accuracy_score(y_test, forest_prediction))

# Feature Importance
importance = pd.DataFrame({"Feature": X.columns, "Importance": forest.feature_importances_})
importance = importance.sort_values(by="Importance", ascending=False)
print("\nMost Important Features:\n", importance)

# Save Outputs
joblib.dump(forest, "employee_prediction_model.pkl")
joblib.dump(clf_scaler, "standard_scaler.pkl")
print("\nModel Saved Successfully")

df["Prediction"] = forest.predict(clf_scaler.transform(X))
top = df[df["Prediction"] == 1]
print("\nEmployees Predicted as High Performance:\n", top[["Employee_ID", "Name", "Department", "Salary", "Performance_Rating"]])

# Increment Recommendation Workflow
def calculate_increment(rating):
    if rating >= 4.8: return "15 %"
    elif rating >= 4.5: return "10 %"
    elif rating >= 4.0: return "7 %"
    elif rating >= 3.5: return "5 %"
    else: return "No Increment"

df["Recommended Increment"] = df["Performance_Rating"].apply(calculate_increment)
print("\nIncrement Recommendation:\n", df[["Employee_ID", "Name", "Performance_Rating", "Recommended Increment"]])

df.to_csv("Prediction_Report.csv", index=False)
print("\nPrediction Report Saved Successfully")

# Deployment Inference Example
loaded_model = joblib.load("employee_prediction_model.pkl")
loaded_scaler = joblib.load("standard_scaler.pkl")

sample = [[30, 1, 2, 65000, 5, 1, 1, 3, 4, 10, 40]]
sample_scaled = loaded_scaler.transform(sample)
result = loaded_model.predict(sample_scaled)

print("\nNew Employee Evaluation: Average Performance Employee")

=== Regression Evaluation ===
MAE : 267140.7208566753
MSE : 123560984215.61038
R2 Score : 0.23400883626069025

========== Logistic Regression ==========
Accuracy : 0.875
Precision : 0.8571428571428571
Recall : 1.0
F1 Score : 0.9230769230769231
Confusion Matrix:
 [[1 1]
 [0 6]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       0.86      1.00      0.92         6

    accuracy                           0.88         8
   macro avg       0.93      0.75      0.79         8
weighted avg       0.89      0.88      0.86         8


========== Decision Tree ==========
Accuracy : 0.75

========== Random Forest ==========
Accuracy : 0.75

Most Important Features:
                Feature  Importance
3             Position    0.196637
10      Overtime_Hours    0.166928
4               Salary    0.128604
9   Projects_Completed    0.126429
0                  Age    0.084343
7                State    0

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [6]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from google.colab import files
uploaded = files.upload()
df = pd.read_csv("Cleaned_Indian_Employee_Data.csv")
numeric_features = ["Age", "Experience_Years", "Projects_Completed", "Overtime_Hours", "Salary", "Performance_Rating"]
df[numeric_features] = df[numeric_features].replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=numeric_features)

encoder = LabelEncoder()
categorical_columns = ["Gender", "Department", "Position", "City", "State", "Education"]
for col in categorical_columns:
    df[col] = encoder.fit_transform(df[col])
X_reg = df[["Age", "Experience_Years", "Projects_Completed", "Overtime_Hours"]]
y_reg = df["Salary"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

reg_scaler = StandardScaler()
X_train_reg = reg_scaler.fit_transform(X_train_reg)
X_test_reg = reg_scaler.transform(X_test_reg)
df["Performance"] = np.where(df["Performance_Rating"] >= 4, 1, 0)
X = df.drop(["Employee_ID", "Name", "Joining_Date", "Performance_Rating", "Performance"], axis=1)
y = df["Performance"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

clf_scaler = StandardScaler()
X_train = clf_scaler.fit_transform(X_train)
X_test = clf_scaler.transform(X_test)
regresson = LinearRegression()
regresson.fit(X_train_reg, y_train_reg)
salary_prediction = regresson.predict(X_test_reg)

print("=== Regression Evaluation ===")
print("MAE :", mean_absolute_error(y_test_reg, salary_prediction))
print("MSE :", mean_squared_error(y_test_reg, salary_prediction))
print("R2 Score :", r2_score(y_test_reg, salary_prediction))


log_model = LogisticRegression()
log_model.fit(X_train, y_train)
log_prediction = log_model.predict(X_test)

print("\n========== Logistic Regression ==========")
print("Accuracy :", accuracy_score(y_test, log_prediction))
print("Precision :", precision_score(y_test, log_prediction))
print("Recall :", recall_score(y_test, log_prediction))
print("F1 Score :", f1_score(y_test, log_prediction))
print("Confusion Matrix:\n", confusion_matrix(y_test, log_prediction))
print("\nClassification Report:\n", classification_report(y_test, log_prediction))

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
tree_prediction = tree.predict(X_test)

print("\n========== Decision Tree ==========")
print("Accuracy :", accuracy_score(y_test, tree_prediction))

forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)
forest_prediction = forest.predict(X_test)

print("Accuracy :", accuracy_score(y_test, forest_prediction))

importance = pd.DataFrame({"Feature": X.columns, "Importance": forest.feature_importances_})
importance = importance.sort_values(by="Importance", ascending=False)
print("\nMost Important Features:\n", importance)

joblib.dump(forest, "employee_prediction_model.pkl")
joblib.dump(clf_scaler, "standard_scaler.pkl")
print("\nModel Saved Successfully")

df["Prediction"] = forest.predict(clf_scaler.transform(X))
top = df[df["Prediction"] == 1]
print("\nEmployees Predicted as High Performance:\n", top[["Employee_ID", "Name", "Department", "Salary", "Performance_Rating"]])

def calculate_increment(rating):
    if rating >= 4.8: return "15 %"
    elif rating >= 4.5: return "10 %"
    elif rating >= 4.0: return "7 %"
    elif rating >= 3.5: return "5 %"
    else: return "No Increment"

df["Recommended Increment"] = df["Performance_Rating"].apply(calculate_increment)
print("\nIncrement Recommendation:\n", df[["Employee_ID", "Name", "Performance_Rating", "Recommended Increment"]])

df.to_csv("Prediction_Report.csv", index=False)
print("\nPrediction Report Saved Successfully")

loaded_model = joblib.load("employee_prediction_model.pkl")
loaded_scaler = joblib.load("standard_scaler.pkl")

sample = [[30, 1, 2, 65000, 5, 1, 1, 3, 4, 10, 40]]
sample_scaled = loaded_scaler.transform(sample)
result = loaded_model.predict(sample_scaled)

if result[0] == 1:
    print("\nNew Employee Evaluation: High Performance Employee")
else:
    print("\nNew Employee Evaluation: Average Performance Employee")

running = True

Saving Cleaned_Indian_Employee_Data.csv to Cleaned_Indian_Employee_Data.csv
=== Regression Evaluation ===
MAE : 267140.7208566753
MSE : 123560984215.61038
R2 Score : 0.23400883626069025

========== Logistic Regression ==========
Accuracy : 0.875
Precision : 0.8571428571428571
Recall : 1.0
F1 Score : 0.9230769230769231
Confusion Matrix:
 [[1 1]
 [0 6]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       0.86      1.00      0.92         6

    accuracy                           0.88         8
   macro avg       0.93      0.75      0.79         8
weighted avg       0.89      0.88      0.86         8


========== Decision Tree ==========
Accuracy : 0.75
Accuracy : 0.75

Most Important Features:
                Feature  Importance
3             Position    0.196637
10      Overtime_Hours    0.166928
4               Salary    0.128604
9   Projects_Completed    0.126429
0                  Age 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
